# Day 50 — Practice solutions

Run the teaching notebook first for the final comparison task. Numeric examples are hand-checkable. Read the reasoning before running the code.

## 1 · One SGD step

$\theta_1=3-.1(2)=2.8$. The gradient is a local direction of increasing loss; its negative is the first-order downhill direction.

In [ ]:
theta=3.; theta-=.1*2.; print(theta)

## 2 · Learning-rate scale

At $.01$: $1-.01(-4)=1.04$. At $.2$: $1-.2(-4)=1.8$. The second step is twenty times farther. A larger step can overshoot; distance alone does not prove a better loss.

In [ ]:
print(1-.01*(-4),1-.2*(-4))

## 3 · Two PyTorch momentum steps

PyTorch initializes $v_1=g_1=2$, so $\theta_1=2.8$. Next $v_2=.9(2)+1=2.8$, so $\theta_2=2.8-.1(2.8)=2.52$.

In [ ]:
v1=2.;v2=.9*v1+1.;print(v1,v2,3-.1*v1-.1*v2)

## 4 · Oscillation and momentum

$v_1=2$; $v_2=.9(2)-2=-.2$; $v_3=.9(-.2)+2=1.82$. Buffering modifies the second step substantially. It can smooth some alternating gradients, but the step still depends on learning rate and local geometry.

In [ ]:
v=None
for g in [2.,-2.,2.]:
    v=g if v is None else .9*v+g
    print(round(v,3))

## 5 · First Adam step

$m_1=.2$, $v_1=.004$, $\hat m_1=.2/.1=2$, $\hat v_1=.004/.001=4$. Change $\Delta\theta=-.1(2)/(\sqrt4+\epsilon)\approx-.1$.

In [ ]:
import math
print(.2/(1-.9),.004/(1-.999),-.1*2/(math.sqrt(4)+1e-8))

## 6 · Why bias correction?

The moments start from zero, so $m_1=(1-.9)2=.2$, far below the actual first gradient. Dividing by $1-.9=.1$ yields $\hat m_1=2$. Correction is strongest early and tends toward one as $t$ grows.

## 7 · AdamW shrinkage

Shrinkage alone is $\eta\lambda\theta=.1(.01)(3)=.003$; the parameter becomes $2.997$ before accounting for the adaptive gradient contribution. AdamW applies decay directly to the parameter; L2 added to the gradient would enter Adam's moment estimates and be adaptively scaled.

In [ ]:
print(3*(1-.1*.01))

## 8 · Step schedule timeline

Epochs 1 and 12 use $.12$; 13 and 24 use $.06$; 25 uses $.03$. Calls occur after each epoch, so the new rate is for the next epoch.

In [ ]:
print({e:.12*.5**((e-1)//12) for e in [1,12,13,24,25]})

## 9 · Cosine midpoint

$\eta(0)=.12$, $\eta(20)=(.12+.005)/2=.0625$, $\eta(40)=.005$. `T_max` counts scheduler calls. It represents epochs only if you call `.step()` once per epoch.

In [ ]:
import math
for t in [0,20,40]: print(t,.005+.5*(.12-.005)*(1+math.cos(math.pi*t/40)))

## 10 · Fix the loop

Put `zero_grad → forward → loss → backward → optimizer.step()` inside the batch loop. After every batch has completed, record the epoch result and call `scheduler.step()` once **at the epoch end**. Log `optimizer.param_groups[0]["lr"]` before the batch updates to capture the rate actually used during that epoch.

## 11 · Plateau and OneCycle

After computing the held-out loss, `plateau.step(val_loss)`; after **each** `optimizer.step()` in a batch loop, `onecycle.step()`. Plateau needs a validation metric for the chosen `mode`. OneCycle needs `total_steps` or `epochs × steps_per_epoch`. Its rate may also modify momentum.

## 12 · Compare and resume

Use `optimizer_runs` and `schedule_runs` from the teaching notebook; inspect their `history["val"][-1]` and `history["lr"]`, then state the observed winner only for your seeded run and these chosen hyperparameters. `model.state_dict()` omits momentum/Adam buffers and scheduler progress, so save the optimizer and scheduler states too. Exact mid-epoch replay also needs sampler and RNG state.

In [ ]:
for group in [optimizer_runs,schedule_runs]:
    for name,(_,h) in group.items():
        print(name,"final validation",round(h["val"][-1],4),"rate",round(h["lr"][-1],5))
# checkpoint = {"model": model.state_dict(),
#               "optimizer": optimizer.state_dict(),
#               "scheduler": scheduler.state_dict(), "epoch": epoch}